# AnnData Object Construction

The `AnnData object` is constructed from the `Seurat object's` extracted data.

## Load Libraries

In [4]:
# Standard imports
import datetime
import os

# Third-party imports
import anndata as ad
import pandas as pd
from scipy.io import mmread
from scipy.sparse import csr_matrix

## Load Parameters

In [91]:
data_dir = input("Enter data directory: ")

# Implement fall back
if data_dir == "":
    data_dir = "../data/adata_conversion/data_extraction_17-02-2025_14-51"

print(f"💾 Data extracted from: {data_dir}")

💾 Data extracted from: ../data/adata_conversion/data_extraction_17-02-2025_14-51


In [92]:
out_dir = input("Enter output directory: ")

# Implement fall back
if out_dir == "":
    out_dir = "../data/h5ad_objects"

print(f"📁 Data extracted from: {out_dir}")

📁 Data extracted from: ../data/h5ad_objects


## Load Data

The `AnnData object` is built around a count matrix, which is often represented in a sparse representation. This means that only RNA counts greater than 0 are stored as actual values.

In [93]:
# Load sparse count matrix from "mtx"
sparse_matrix = mmread(os.path.join(data_dir, "seurat_counts.mtx"))
sparse_matrix

<COOrdinate sparse matrix of dtype 'int64'
	with 16181234 stored elements and shape (970, 67785)>

Convert the matrix to the `CSR` format to improve performance.  
Improves run time since most operations are run on samples instead of features.  

&rarr; `Seurat` matrix needs to be transposed (other orientation)

In [94]:
# Convert to CSR format
csr_matrix = csr_matrix(sparse_matrix)
csr_matrix_t = csr_matrix.transpose().tocsr()
csr_matrix_t

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 16181234 stored elements and shape (67785, 970)>

Row and column names must be loaded separately because they are not saved in the `MMatrix object`.

In [95]:
# Load row and column names from "csv"
# Important: AnnData row names are Seurat column names
# Same for AnnData column names and Seurat row names
row_names = pd.read_csv(
    os.path.join(data_dir, "seurat_counts_col_names.csv"), header=None
)
col_names = pd.read_csv(
    os.path.join(data_dir, "seurat_counts_row_names.csv"), header=None
)

# Sanity check
if (
    csr_matrix_t.shape[0] == row_names.shape[0]
    and csr_matrix_t.shape[1] == col_names.shape[0]
):
    print("ℹ️ Row and column names match matrix dimensions")

ℹ️ Row and column names match matrix dimensions


Metadata is stored in a matrix that can be initialized to the `obs` parameter during creation.

In [96]:
# Load metadata from "csv"
obs_data = pd.read_csv(os.path.join(data_dir, "seurat_metadata.csv"))

# Add row names as column
# Column can be set as index
obs_data["sample_names"] = row_names[0].values
obs_data = obs_data.set_index("sample_names")
obs_data.index.name = None  # Remove index title

# Rename columns
obs_data = obs_data.rename(
    columns={
        "nCount_Nanostring": "counts_per_cell",
        "nFeature_Nanostring": "non_zero_features",
        "cell_ID": "cell_id",
        "Slide_name": "slide",
    }
)

obs_data

,counts_per_cell,non_zero_features,cell_id,condition,slide,tissue,fov
liver1_3_1,450,181,c_1_1_3,healthy,S2,liver1,1_liver1
liver1_4_1,780,241,c_1_1_4,healthy,S2,liver1,1_liver1
liver1_5_1,752,260,c_1_1_5,healthy,S2,liver1,1_liver1
liver1_7_1,725,323,c_1_1_7,healthy,S2,liver1,1_liver1
liver1_8_1,395,142,c_1_1_8,healthy,S2,liver1,1_liver1
...,...,...,...,...,...,...,...
liver2_531_45,579,222,c_2_45_531,cirrhosis,S3,liver2,45_liver2
liver2_532_45,1015,270,c_2_45_532,cirrhosis,S3,liver2,45_liver2
liver2_533_45,954,264,c_2_45_533,cirrhosis,S3,liver2,45_liver2
liver2_534_45,2247,418,c_2_45_534,cirrhosis,S3,liver2,45_liver2


Compute if a gene (feature) has zero expression across all samples and save that as metadata.

In [97]:
# Convert to CSC to improve computing
csc_matrix = csr_matrix_t.tocsc()
csc_matrix

<Compressed Sparse Column sparse matrix of dtype 'int64'
	with 16181234 stored elements and shape (67785, 970)>

In [98]:
# Initialize data frame
var_data = pd.DataFrame(index=col_names[0].values)
var_data

""
Abca2
Abi1
Abi2
Abl1
Ace2
...
NegPrb6
NegPrb7
NegPrb8
NegPrb9


In [99]:
counts_per_gene = csc_matrix.sum(axis=0)
counts_per_gene.shape

(1, 970)

In [100]:
var_data["counts_per_gene"] = counts_per_gene.A1
var_data

,counts_per_gene
Abca2,18947
Abi1,48137
Abi2,9686
Abl1,16608
Ace2,5576
...,...
NegPrb6,4902
NegPrb7,8887
NegPrb8,4277
NegPrb9,2622


## Object Construction

In [101]:
adata = ad.AnnData(X=csr_matrix_t, obs=obs_data, var=var_data)
adata

AnnData object with n_obs × n_vars = 67785 × 970
    obs: 'counts_per_cell', 'non_zero_features', 'cell_id', 'condition', 'slide', 'tissue', 'fov'
    var: 'counts_per_gene'

In [102]:
# Verify obs names
adata.obs_names

Index(['liver1_3_1', 'liver1_4_1', 'liver1_5_1', 'liver1_7_1', 'liver1_8_1',
       'liver1_9_1', 'liver1_10_1', 'liver1_11_1', 'liver1_12_1',
       'liver1_13_1',
       ...
       'liver2_526_45', 'liver2_527_45', 'liver2_528_45', 'liver2_529_45',
       'liver2_530_45', 'liver2_531_45', 'liver2_532_45', 'liver2_533_45',
       'liver2_534_45', 'liver2_535_45'],
      dtype='object', length=67785)

In [103]:
# Verfiy var names
adata.var_names

Index(['Abca2', 'Abi1', 'Abi2', 'Abl1', 'Ace2', 'Acer3', 'Acta2', 'Ada',
       'Adam10', 'Adam22',
       ...
       'NegPrb1', 'NegPrb2', 'NegPrb3', 'NegPrb4', 'NegPrb5', 'NegPrb6',
       'NegPrb7', 'NegPrb8', 'NegPrb9', 'NegPrb10'],
      dtype='object', length=970)

## Save Object

In [104]:
# Save current moment
now = datetime.datetime.now().strftime("%d-%m-%Y_%H-%M")
now

'17-02-2025_16-32'

In [106]:
adata

AnnData object with n_obs × n_vars = 67785 × 970
    obs: 'counts_per_cell', 'non_zero_features', 'cell_id', 'condition', 'slide', 'tissue', 'fov'
    var: 'counts_per_gene'

In [105]:
adata.write(os.path.join(out_dir, f"merged_liver_{now}.h5ad"))

## Create Subset for Testing

In [2]:
data_file = input("Enter file name:")
data_file

'merged_liver_17-02-2025_16-32.h5ad'

In [5]:
cdata = ad.read_h5ad(os.path.join("../data/h5ad_objects", data_file))
cdata

AnnData object with n_obs × n_vars = 67785 × 970
    obs: 'counts_per_cell', 'non_zero_features', 'cell_id', 'condition', 'slide', 'tissue', 'fov'
    var: 'counts_per_gene'
    uns: 'log1p'

In [17]:
cdata = cdata[:100, :100]
cdata.raw = cdata.raw[:100, :100].to_adata()
cdata

AnnData object with n_obs × n_vars = 100 × 100
    obs: 'counts_per_cell', 'non_zero_features', 'cell_id', 'condition', 'slide', 'tissue', 'fov'
    var: 'counts_per_gene'
    uns: 'log1p'

In [20]:
cdata.write("../data/h5ad_objects/test_subset.h5ad")